In [1]:
# ============================================
# 정당별 관심사(분기 기반) 전처리/집계 - 전체 코드
# 입력:
#   - *_minutes_speeches.xlsx (발언)
#   - (분류2 결과) agenda_labeled*.csv (안건 라벨 포함)
# 출력(DB 적재용 CSV):
#   - party_l2_quarter.csv
#   - party_l3_quarter.csv
#   - viz_party_l2_quarter_metrics.csv
#   - viz_party_l3_quarter_metrics.csv
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

# -----------------------------
# 0) 경로 설정
# -----------------------------
BASE_DIR = Path(r"C:\Users\user\Desktop\코드\데이터수집\data")

TREND_OUT_DIR = BASE_DIR / "_outputs_trend"              # 분류2 결과가 보통 여기로 저장됨
OUT_DIR       = BASE_DIR / "_outputs_party_interest"     # 정당별 산출물 저장
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) 유틸
# -----------------------------
def parse_korean_date(x: str):
    if pd.isna(x):
        return pd.NaT
    s = str(x).strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", s)
    if not m:
        return pd.NaT
    y, mo, d = map(int, m.groups())
    try:
        return pd.Timestamp(y, mo, d)
    except:
        return pd.NaT

def pick_first_existing_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def extract_int_list(x):
    """ '1,2,3' / '[1, 2]' / '1 2 3' 등에서 정수 리스트 추출 """
    if pd.isna(x):
        return []
    s = str(x)
    nums = re.findall(r"\d+", s)
    return [int(n) for n in nums] if nums else []

def add_period_cols(df, date_col="date_dt"):
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out = out[out[date_col].notna()].copy()
    out["year"] = out[date_col].dt.year.astype(int)
    out["quarter"] = out[date_col].dt.quarter.astype(int)
    out["period"] = out["year"].astype(str) + "-Q" + out["quarter"].astype(str)
    return out

def find_latest_agenda_labeled(base_dir: Path):
    # 후보: _outputs_trend 안에 agenda_labeled*.csv
    cands = list((base_dir / "_outputs_trend").glob("agenda_labeled*.csv"))
    # 혹시 다른 폴더에 있으면 전체에서 보조 탐색(느릴 수 있음)
    if not cands:
        cands = list(base_dir.rglob("agenda_labeled*.csv"))

    # 임시파일/숨김 제외
    cands = [p for p in cands if p.is_file() and not p.name.startswith("~$")]

    if not cands:
        raise FileNotFoundError(f"agenda_labeled*.csv를 찾지 못했습니다. base_dir={base_dir}")

    # v3 > v2 > 그 외, 그리고 최신 수정시간
    def score(p: Path):
        name = p.name.lower()
        v = 0
        if "v3" in name: v += 30
        if "v2" in name: v += 20
        if "latest" in name: v += 10
        return (v, p.stat().st_mtime)

    cands = sorted(cands, key=score, reverse=True)
    return cands[0], cands

# -----------------------------
# 2) agenda_labeled 로드 (분류2 결과)
# -----------------------------
agenda_path, all_cands = find_latest_agenda_labeled(BASE_DIR)
print("[AGENDA_LABELED] pick:", agenda_path)
# print("[AGENDA_LABELED] candidates:", [p.name for p in all_cands[:10]])

agenda = pd.read_csv(agenda_path, encoding="utf-8-sig")

# 컬럼명 정규화 (l2/l3 둘 다 수용)
L2_COL = "label_l2" if "label_l2" in agenda.columns else ("l2" if "l2" in agenda.columns else None)
L3_COL = "label_l3" if "label_l3" in agenda.columns else ("l3" if "l3" in agenda.columns else None)
BASE_COL = "base" if "base" in agenda.columns else None

if L2_COL is None or L3_COL is None or BASE_COL is None:
    raise RuntimeError(
        "agenda_labeled에 필요한 컬럼이 없습니다. 필요: base + (label_l2/l2) + (label_l3/l3)\n"
        f"현재 컬럼: {agenda.columns.tolist()}"
    )

# item_order_int 생성
if "item_order_int" not in agenda.columns:
    if "item_order" not in agenda.columns:
        raise RuntimeError("agenda_labeled에 item_order 또는 item_order_int가 없습니다.")
    agenda["item_order_int"] = pd.to_numeric(agenda["item_order"], errors="coerce").round(0).astype("Int64")

# meeting_key 없으면 생성 (가능한 컬럼으로 조합)
if "meeting_key" not in agenda.columns:
    # 최소 session, meeting_no, date/date_dt가 있어야 함
    sess_col = pick_first_existing_col(agenda, ["session_num","session","session_no"])
    meet_col = pick_first_existing_col(agenda, ["meeting_no","meetingNo","meeting"])
    date_col = pick_first_existing_col(agenda, ["date_dt","date"])
    if sess_col is None or meet_col is None or date_col is None:
        raise RuntimeError("agenda_labeled에 meeting_key가 없고, 생성에 필요한 session/meeting_no/date도 부족합니다.")
    agenda["date_dt"] = pd.to_datetime(agenda[date_col], errors="coerce")
    agenda["meeting_key"] = (
        pd.to_numeric(agenda[sess_col], errors="coerce").fillna(-1).astype(int).astype(str)
        + "_" + agenda[meet_col].astype(str)
        + "_" + agenda["date_dt"].dt.strftime("%Y-%m-%d")
    )

agenda = agenda[agenda["item_order_int"].notna()].copy()

agenda_key = (
    agenda[["meeting_key","item_order_int", BASE_COL, L2_COL, L3_COL]]
    .drop_duplicates(subset=["meeting_key","item_order_int"])
    .rename(columns={BASE_COL:"base", L2_COL:"label_l2", L3_COL:"label_l3"})
)
print("[AGENDA_KEY]", agenda_key.shape)

# -----------------------------
# 3) minutes_speeches 로드
# -----------------------------
def load_all_minutes_speeches(base_dir: Path):
    files = sorted(base_dir.rglob("*_minutes_speeches.xlsx"))
    files = [fp for fp in files if not fp.name.startswith("~$")]
    if not files:
        raise RuntimeError("minutes_speeches 파일을 찾지 못했습니다. 경로/패턴 확인 필요")

    dfs = []
    bad = []
    for fp in files:
        try:
            df = pd.read_excel(fp, engine="openpyxl")
            df["source_file"] = fp.name
            df["session_dir"] = fp.parent.name
            dfs.append(df)
        except Exception as e:
            bad.append((str(fp), str(e)))

    out = pd.concat(dfs, ignore_index=True)
    print(f"[SPEECHES] loaded: {out.shape} | files={len(files)} | bad={len(bad)}")
    if bad:
        print("[SPEECHES] bad examples:", bad[:3])
    return out

speeches = load_all_minutes_speeches(BASE_DIR)

party_col   = pick_first_existing_col(speeches, ["party","정당","소속정당"])
text_col    = pick_first_existing_col(speeches, ["speech_text","talk","content","utterance","speech","text","발언내용","대화내용","발언"])
speaker_col = pick_first_existing_col(speeches, ["speaker_name","name","speaker","발언자","의원명","위원명"])
orders_col  = pick_first_existing_col(speeches, ["agenda_item_orders","item_orders","agenda_orders","안건번호","안건목록","item_order_list"])

need = {"party":party_col, "text":text_col, "orders":orders_col}
if any(v is None for v in need.values()):
    raise RuntimeError(f"speeches에서 필요한 컬럼 탐지 실패: {need}\n현재 컬럼: {speeches.columns.tolist()}")

# meeting_key 생성
speeches["date_dt"] = speeches.get("date", np.nan).map(parse_korean_date)
sess_col = pick_first_existing_col(speeches, ["session_num","session","session_no"])
meet_col = pick_first_existing_col(speeches, ["meeting_no","meetingNo","meeting"])

if sess_col is None or meet_col is None:
    raise RuntimeError("speeches에서 session/meeting_no 컬럼을 찾지 못했습니다.")

speeches[sess_col] = pd.to_numeric(speeches[sess_col], errors="coerce")
speeches[meet_col] = speeches[meet_col].astype(str)

speeches["meeting_key"] = (
    speeches[sess_col].fillna(-1).astype(int).astype(str)
    + "_" + speeches[meet_col]
    + "_" + speeches["date_dt"].dt.strftime("%Y-%m-%d")
)

# -----------------------------
# 4) 안건번호 explode -> agenda_key와 매핑
# -----------------------------
speeches["item_order_int_list"] = speeches[orders_col].map(extract_int_list)
spx = speeches.explode("item_order_int_list").rename(columns={"item_order_int_list":"item_order_int"})
spx["item_order_int"] = pd.to_numeric(spx["item_order_int"], errors="coerce").astype("Int64")
spx = spx[spx["item_order_int"].notna()].copy()

mapped = spx.merge(agenda_key, on=["meeting_key","item_order_int"], how="inner")

mapped = mapped.rename(columns={party_col:"party", text_col:"speech_text"})
mapped["party"] = mapped["party"].astype(str).str.strip().replace({"": "미분류"})
mapped["text_len"] = mapped["speech_text"].astype(str).str.len()

print("[MAPPED]", mapped.shape, "| meetings:", mapped["meeting_key"].nunique())

# -----------------------------
# 5) 분기 컬럼 부여
# -----------------------------
mapped_q = add_period_cols(mapped, "date_dt")
print("[PERIODS]", mapped_q["period"].nunique(), "| range:", mapped_q["period"].min(), "~", mapped_q["period"].max())

# -----------------------------
# 6) 분기×정당×L2/L3 집계 (DB 적재용)
# -----------------------------
# meeting_count: 해당 분기에 해당 정당이 해당 라벨이 등장한 "서로 다른 회의" 수
# mention_count: 매핑 행 수(발언-안건 매핑 건수)

# L2
uniq_meeting_party_l2 = mapped_q.drop_duplicates(subset=["period","meeting_key","party","label_l2"])
l2_meeting_cnt = uniq_meeting_party_l2.groupby(["period","party","label_l2"], as_index=False).size().rename(columns={"size":"meeting_count"})
l2_mention_cnt = mapped_q.groupby(["period","party","label_l2"], as_index=False).size().rename(columns={"size":"mention_count"})
party_l2_q = l2_meeting_cnt.merge(l2_mention_cnt, on=["period","party","label_l2"], how="left")
party_l2_q["mention_count"] = party_l2_q["mention_count"].fillna(0).astype(int)

# L3
uniq_meeting_party_l3 = mapped_q.drop_duplicates(subset=["period","meeting_key","party","label_l3"])
l3_meeting_cnt = uniq_meeting_party_l3.groupby(["period","party","label_l3"], as_index=False).size().rename(columns={"size":"meeting_count"})
l3_mention_cnt = mapped_q.groupby(["period","party","label_l3"], as_index=False).size().rename(columns={"size":"mention_count"})
party_l3_q = l3_meeting_cnt.merge(l3_mention_cnt, on=["period","party","label_l3"], how="left")
party_l3_q["mention_count"] = party_l3_q["mention_count"].fillna(0).astype(int)

# 저장
PARTY_L2_Q_CSV = OUT_DIR / "party_l2_quarter.csv"
PARTY_L3_Q_CSV = OUT_DIR / "party_l3_quarter.csv"
party_l2_q.to_csv(PARTY_L2_Q_CSV, index=False, encoding="utf-8-sig")
party_l3_q.to_csv(PARTY_L3_Q_CSV, index=False, encoding="utf-8-sig")
print("[SAVE]", PARTY_L2_Q_CSV)
print("[SAVE]", PARTY_L3_Q_CSV)

# -----------------------------
# 7) (권장) 분기×정당 보정지표(viz) 생성: L2/L3 둘 다 만들기
# -----------------------------
# 분모: period×party
party_den_q = (
    mapped_q.groupby(["period","party"], as_index=False)
    .agg(
        party_meetings=("meeting_key","nunique"),
        party_total_chars=("text_len","sum"),
    )
)

def build_viz(metrics_df, label_col):
    # metrics_df: period, party, label_col, meeting_count, mention_count
    period_list = sorted(mapped_q["period"].unique().tolist())
    party_list  = sorted(mapped_q["party"].unique().tolist())
    label_list  = sorted(mapped_q[label_col].dropna().unique().tolist())

    grid = pd.MultiIndex.from_product([period_list, party_list, label_list],
                                      names=["period","party",label_col]).to_frame(index=False)

    viz = (
        grid.merge(metrics_df, on=["period","party",label_col], how="left")
            .merge(party_den_q, on=["period","party"], how="left")
    )

    viz["meeting_count"] = viz["meeting_count"].fillna(0).astype(int)
    viz["mention_count"] = viz["mention_count"].fillna(0).astype(int)
    viz["party_meetings"] = viz["party_meetings"].fillna(0).astype(int)
    viz["party_total_chars"] = viz["party_total_chars"].fillna(0).astype(int)

    viz["meeting_presence_rate"] = np.where(
        viz["party_meetings"] > 0,
        (viz["meeting_count"] / viz["party_meetings"]).round(6),
        0.0
    )
    viz["mentions_per_10k_chars"] = np.where(
        viz["party_total_chars"] > 0,
        (viz["mention_count"] / viz["party_total_chars"] * 10000).round(6),
        0.0
    )
    return viz

[AGENDA_LABELED] pick: C:\Users\user\Desktop\코드\데이터수집\data\_outputs_trend\agenda_labeled_v3.csv
[AGENDA_KEY] (9452, 5)
[SPEECHES] loaded: (116306, 16) | files=258 | bad=0
[MAPPED] (994002, 23) | meetings: 224
[PERIODS] 35 | range: 2017-Q3 ~ 2026-Q1
[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\party_l2_quarter.csv
[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\party_l3_quarter.csv


In [2]:
viz_l2 = build_viz(party_l2_q, "label_l2")
viz_l3 = build_viz(party_l3_q, "label_l3")

VIZ_L2_CSV = OUT_DIR / "viz_party_l2_quarter_metrics.csv"
VIZ_L3_CSV = OUT_DIR / "viz_party_l3_quarter_metrics.csv"
viz_l2.to_csv(VIZ_L2_CSV, index=False, encoding="utf-8-sig")
viz_l3.to_csv(VIZ_L3_CSV, index=False, encoding="utf-8-sig")
print("[SAVE]", VIZ_L2_CSV)
print("[SAVE]", VIZ_L3_CSV)

# 미리보기
display(party_l2_q.sort_values(["period","party","meeting_count"], ascending=[True, True, False]).head(10))

[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\viz_party_l2_quarter_metrics.csv
[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\viz_party_l3_quarter_metrics.csv


,period,party,label_l2,meeting_count,mention_count
2,2017-Q3,국민의당,지방재정·기타,3,88
1,2017-Q3,국민의당,정부혁신·행정지원,2,87
0,2017-Q3,국민의당,재난·안전,1,52
3,2017-Q3,더불어민주당,재난·안전,2,86
4,2017-Q3,더불어민주당,정부혁신·행정지원,2,210
5,2017-Q3,더불어민주당,지방재정·기타,2,199
7,2017-Q3,미분류,정부혁신·행정지원,3,400
8,2017-Q3,미분류,지방재정·기타,3,376
6,2017-Q3,미분류,재난·안전,2,315
9,2017-Q3,미분류,지방행정,1,10


In [3]:
viz_l2

,period,party,label_l2,meeting_count,mention_count,party_meetings,party_total_chars,meeting_presence_rate,mentions_per_10k_chars
0,2017-Q3,국민의당,인공지능·디지털 정부,0,0,3,36070,0.000000,0.000000
1,2017-Q3,국민의당,재난·안전,1,52,3,36070,0.333333,14.416413
2,2017-Q3,국민의당,정부혁신·행정지원,2,87,3,36070,0.666667,24.119767
3,2017-Q3,국민의당,지방재정·기타,3,88,3,36070,1.000000,24.397006
4,2017-Q3,국민의당,지방행정,0,0,3,36070,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
2095,2026-Q1,진보당,인공지능·디지털 정부,1,3,1,317216,1.000000,0.094573
2096,2026-Q1,진보당,재난·안전,1,72,1,317216,1.000000,2.269747
2097,2026-Q1,진보당,정부혁신·행정지원,1,46,1,317216,1.000000,1.450116
2098,2026-Q1,진보당,지방재정·기타,1,14,1,317216,1.000000,0.441340


In [4]:
viz_l3

,period,party,label_l3,meeting_count,mention_count,party_meetings,party_total_chars,meeting_presence_rate,mentions_per_10k_chars
0,2017-Q3,국민의당,국가기록물,0,0,3,36070,0.0,0.000000
1,2017-Q3,국민의당,국가적중장기과제추진·지원,0,0,3,36070,0.0,0.000000
2,2017-Q3,국민의당,국민안전행정지원,0,0,3,36070,0.0,0.000000
3,2017-Q3,국민의당,복구지원,0,0,3,36070,0.0,0.000000
4,2017-Q3,국민의당,비상대비,0,0,3,36070,0.0,0.000000
...,...,...,...,...,...,...,...,...,...
8395,2026-Q1,진보당,지방세,1,4,1,317216,1.0,0.126097
8396,2026-Q1,진보당,지방재정,1,10,1,317216,1.0,0.315243
8397,2026-Q1,진보당,지방행정,1,36,1,317216,1.0,1.134873
8398,2026-Q1,진보당,지역균형발전,1,13,1,317216,1.0,0.409815


In [5]:
# ============================================
# [PATCH] L3 집계에 label_l2 포함 (재생성)
# - 기존 mapped_q를 그대로 사용
# - 결과: party_l3_q2 (period, party, label_l2, label_l3, meeting_count, mention_count)
# ============================================

# 1) L3 집계는 반드시 label_l2 + label_l3 같이 묶는다
uniq_meeting_party_l3 = mapped_q.drop_duplicates(
    subset=["period", "meeting_key", "party", "label_l2", "label_l3"]
)
l3_meeting_cnt2 = (
    uniq_meeting_party_l3
    .groupby(["period", "party", "label_l2", "label_l3"], as_index=False)
    .size().rename(columns={"size": "meeting_count"})
)

l3_mention_cnt2 = (
    mapped_q
    .groupby(["period", "party", "label_l2", "label_l3"], as_index=False)
    .size().rename(columns={"size": "mention_count"})
)

party_l3_q2 = l3_meeting_cnt2.merge(
    l3_mention_cnt2,
    on=["period", "party", "label_l2", "label_l3"],
    how="left",
)
party_l3_q2["mention_count"] = party_l3_q2["mention_count"].fillna(0).astype(int)

print("[party_l3_q2]", party_l3_q2.shape)
display(party_l3_q2.head(10))

[party_l3_q2] (1174, 6)


,period,party,label_l2,label_l3,meeting_count,mention_count
0,2017-Q3,국민의당,재난·안전,안전및재난,1,52
1,2017-Q3,국민의당,정부혁신·행정지원,정부혁신,2,87
2,2017-Q3,국민의당,지방재정·기타,지방재정,3,88
3,2017-Q3,더불어민주당,재난·안전,안전및재난,2,86
4,2017-Q3,더불어민주당,정부혁신·행정지원,정부의전,1,12
5,2017-Q3,더불어민주당,정부혁신·행정지원,정부혁신,1,198
6,2017-Q3,더불어민주당,지방재정·기타,지방재정,2,199
7,2017-Q3,미분류,재난·안전,안전및재난,2,315
8,2017-Q3,미분류,정부혁신·행정지원,정부의전,1,23
9,2017-Q3,미분류,정부혁신·행정지원,정부혁신,3,367


In [6]:
# ============================================
# [PATCH] DB 적재용 CSV 저장
# ============================================
# PARTY_L2_Q_CSV = OUT_DIR / "party_l2_quarter.csv"  # 기존과 동일
PARTY_L3_Q2_CSV = OUT_DIR / "party_l3_quarter_with_l2.csv"  # ✅ 새 파일명

# # L2는 그대로 저장(이미 저장했으면 생략 가능)
# party_l2_q.to_csv(PARTY_L2_Q_CSV, index=False, encoding="utf-8-sig")

# L3는 label_l2 포함 버전으로 저장
party_l3_q2.to_csv(PARTY_L3_Q2_CSV, index=False, encoding="utf-8-sig")

print("[SAVE]", PARTY_L2_Q_CSV)
print("[SAVE]", PARTY_L3_Q2_CSV)

[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\party_l2_quarter.csv
[SAVE] C:\Users\user\Desktop\코드\데이터수집\data\_outputs_party_interest\party_l3_quarter_with_l2.csv
